# Advanced Python Boolean Protocol — 20 Problems with Fully Worked Solutions

**Focus:** truth-value testing for custom objects, `__bool__`, fallback to `__len__`, boolean operators, safe API design, and protocol edge cases. **Difficulty:** advanced; problems increase in complexity. **Prerequisites:** Python classes, iteration, exceptions, basic typing.

**How to use:** Run **Kernel → Restart & Run All** on Python 3. Each problem includes a specification, a worked solution, and executable assertions. All examples use only the Python standard library. Code is deliberately verbose where it makes behavior auditable; tests avoid using `assert some_object` when that would obscure the condition being checked.

**Provenance:** The attached lesson establishes the core rules and examples (`Person`, `MyList`, and `Point`). This workbook extends those ideas with original problems on protocol failure modes, operator behavior, collections, iterators, domain models, and testing; the advanced scenarios are extensions rather than claims made in the source.

## Protocol reference (read before solving)

Python determines `bool(obj)` by calling a type-level `obj.__bool__()` if defined; otherwise it consults `obj.__len__()` if defined (zero means false, positive means true); otherwise the object is true. `__bool__` **must return an actual `bool`**, not an integer or any other merely truthy object. `__len__` must return a nonnegative integer representable as a Python sequence length (an integer supporting the index protocol is accepted); invalid lengths fail. Built-in `and`/`or` return an **operand**, not necessarily a boolean, and short-circuit. `not`, `bool`, `if`, `while`, `all`, and `any` evaluate truth.

**Best practices:** If your object models a collection, prefer correct, cheap `__len__` and let emptiness define truth; use `__bool__` only if there is a clearly documented alternative meaning. Don't use `if value` when `0`, `False`, or an empty collection are valid values distinct from missing (`None`). Avoid expensive, state-mutating, or surprising truth checks. If truth has no unambiguous meaning, raise `TypeError` and provide explicit methods instead.

**Study plan:** Predict each assertion's outcome first; implement a solution independently; compare with the worked answer; then alter the test data and re-run all cells.

---

# Part I — Protocol precedence and invariants

Problems 1–4 investigate dispatch precedence, strict return types, legal lengths, and collection performance.

## Problem 01 — Truthiness dispatch tracer

**Challenge.** Implement three instrumented classes: an object with neither hook, an object with only `__len__`, and an object with both hooks. Record which hooks were invoked, and prove `__bool__` takes precedence even when `__len__` says the opposite.

**Acceptance criteria / edge cases.** Test empty vs nonempty length; conflicting hook results; calls must be exactly as expected; default objects are truthy.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 01

In [1]:
class P01Default:
    pass


class P01LengthOnly:
    def __init__(self, length: int, calls: list[str]) -> None:
        self.length = length
        self.calls = calls

    def __len__(self) -> int:
        self.calls.append("len")
        return self.length


class P01Both(P01LengthOnly):
    def __init__(self, length: int, answer: bool, calls: list[str]) -> None:
        super().__init__(length, calls)
        self.answer = answer

    def __bool__(self) -> bool:
        self.calls.append("bool")
        return self.answer

**Verification 01.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [2]:
assert bool(P01Default()) is True
calls_1 = []
assert bool(P01LengthOnly(0, calls_1)) is False
assert calls_1 == ["len"]
calls_2 = []
assert bool(P01LengthOnly(3, calls_2)) is True
assert calls_2 == ["len"]
calls_3 = []
assert bool(P01Both(0, True, calls_3)) is True
assert calls_3 == ["bool"]
calls_4 = []
assert bool(P01Both(50, False, calls_4)) is False
assert calls_4 == ["bool"]
print("Problem 01: all checks passed")

Problem 01: all checks passed


## Problem 02 — Fix the incorrect Point boolean

**Challenge.** Reproduce the source lesson's subtle bug: returning `self.x or self.y` from `__bool__`. Catch the **expected** exception without hiding unrelated errors. Then write a correct immutable `Point` where only `(0, 0)` is false.

**Acceptance criteria / edge cases.** Verify the broken zero and nonzero cases both fail with `TypeError` (an `int` is not a `bool`); verify both corrected coordinates and negative coordinates. Use explicit comparisons rather than assuming numbers are returned as booleans.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 02

In [3]:
from dataclasses import dataclass


class P02BrokenPoint:
    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y

    def __bool__(self) -> bool:
        return self.x or self.y  # BUG: this returns int, not bool.


@dataclass(frozen=True)
class P02Point:
    x: int
    y: int

    def __bool__(self) -> bool:
        return self.x != 0 or self.y != 0

**Verification 02.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [4]:
for coordinates in ((0, 0), (3, 0), (0, -8)):
    broken = P02BrokenPoint(*coordinates)
    try:
        bool(broken)
    except TypeError as exc:
        assert "__bool__" in str(exc)
    else:
        raise AssertionError("Expected TypeError from a non-bool return")

for coordinates, expected in [((0, 0), False), ((3, 0), True), ((0, -8), True)]:
    point = P02Point(*coordinates)
    assert bool(point) is expected
    assert type(point.__bool__()) is bool
assert P02Point(1, 2) == P02Point(1, 2)
print("Problem 02: all checks passed")

Problem 02: all checks passed


## Problem 03 — Validate the `__len__` contract

**Challenge.** Build objects that return lengths of `0`, `7`, `-1`, `2.5`, and `sys.maxsize + 1`. Predict which work for `bool()` and for `len()`, and why. Do not rely on particular full exception messages.

**Acceptance criteria / edge cases.** Expect `ValueError` for negative, `TypeError` for non-indexable float, and `OverflowError` for overlarge lengths on CPython-supported Python versions; use a small helper that verifies exact exception type rather than a broad catch.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 03

In [5]:
import sys


class P03Length:
    def __init__(self, answer: object) -> None:
        self.answer = answer

    def __len__(self) -> int:
        return self.answer


def p03_expect_exception(error_type: type[Exception], action) -> None:
    try:
        action()
    except error_type as exc:
        if type(exc) is not error_type:
            raise AssertionError(f"Wrong exception subclass: {type(exc)!r}")
    else:
        raise AssertionError(f"Expected {error_type.__name__}")

**Verification 03.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [6]:
for length, expected in [(0, False), (7, True)]:
    obj = P03Length(length)
    assert bool(obj) is expected
    assert len(obj) == length
for action in (bool, len):
    p03_expect_exception(ValueError, lambda action=action: action(P03Length(-1)))
    p03_expect_exception(TypeError, lambda action=action: action(P03Length(2.5)))
    p03_expect_exception(OverflowError, lambda action=action: action(P03Length(sys.maxsize + 1)))
# A huge Python integer is legal in isolation, but not a sequence length.
print("Problem 03: all checks passed")

Problem 03: all checks passed


## Problem 04 — Fast truthiness for a mutable collection

**Challenge.** Implement a bag supporting `add`, `discard_one`, `__len__`, and iteration. Emptiness must be correct after every mutation, and the length must be available without rescanning the bag. Compare it with a deliberately slow `__bool__` that walks every item.

**Acceptance criteria / edge cases.** `discard_one` must remove only one equal item, return a `bool`, and leave the bag untouched when the item is absent. A list-backed bag gives O(1) length and O(n) deletion; don't make false complexity claims.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 04

In [7]:
from typing import Generic, Iterator, TypeVar

P04T = TypeVar("P04T")


class P04Bag(Generic[P04T]):
    def __init__(self) -> None:
        self._items: list[P04T] = []
        self.length_checks = 0

    def add(self, item: P04T) -> None:
        self._items.append(item)

    def discard_one(self, item: P04T) -> bool:
        try:
            self._items.remove(item)
        except ValueError:
            return False
        return True

    def __len__(self) -> int:
        self.length_checks += 1
        return len(self._items)  # O(1) for Python lists.

    def __iter__(self) -> Iterator[P04T]:
        return iter(self._items)


class P04SlowBag(P04Bag[P04T]):
    def __bool__(self) -> bool:
        return any(True for _ in self._items)  # Additional traversal work.

**Verification 04.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [8]:
bag = P04Bag[int]()
assert bool(bag) is False and bag.length_checks == 1
bag.add(0)
bag.add(0)
assert bool(bag) is True
assert len(bag) == 2
assert bag.discard_one(0) is True
assert list(bag) == [0]
assert bag.discard_one(99) is False
assert len(bag) == 1
assert bag.discard_one(0) is True
assert bool(bag) is False
assert bag.length_checks == 6  # Three bool calls + three len calls.
slow = P04SlowBag[int]()
slow.add(0)
assert bool(slow) is True
assert slow.length_checks == 0  # A subclass __bool__ overrides __len__.
print("Problem 04: all checks passed")

Problem 04: all checks passed


---

# Part II — Operators and missing-data semantics

Problems 5–8 cover operand selection, short-circuiting, the difference between false and missing, and three-valued state.

## Problem 05 — Prove `and`/`or` return operands

**Challenge.** Create objects whose `__bool__` logs evaluations. Verify return-by-identity of `and`/`or`, and show that the second operand's boolean hook need not run at all. Contrast this with `not` returning a real boolean.

**Acceptance criteria / edge cases.** Explicitly test both truth values of the first operand; log should contain only the left operand for each two-operand expression.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 05

In [9]:
class P05Probe:
    def __init__(self, name: str, answer: bool, events: list[str]) -> None:
        self.name = name
        self.answer = answer
        self.events = events

    def __bool__(self) -> bool:
        self.events.append(self.name)
        return self.answer


def p05_evaluate(left_answer: bool):
    events: list[str] = []
    left = P05Probe("left", left_answer, events)
    right = P05Probe("right", True, events)
    and_result = left and right
    and_events = events.copy()
    events.clear()
    or_result = left or right
    or_events = events.copy()
    events.clear()
    not_result = not left
    return left, right, and_result, or_result, not_result, and_events, or_events, events

**Verification 05.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [10]:
left, right, a, o, negated, and_log, or_log, not_log = p05_evaluate(False)
assert a is left and o is right and negated is True
assert and_log == or_log == not_log == ["left"]
left, right, a, o, negated, and_log, or_log, not_log = p05_evaluate(True)
assert a is right and o is left and negated is False
assert and_log == or_log == not_log == ["left"]
assert type(negated) is bool
print("Problem 05: all checks passed")

Problem 05: all checks passed


## Problem 06 — Fix falsy-but-valid configuration

**Challenge.** A configuration accepts `timeout=0` (no waiting), `label=""` (intentionally blank), and `items=[]` (intentionally empty). Replace dangerous `x or fallback` defaults while preserving these explicit values. Distinguish omitted fields (`None`) from supplied falsy values.

**Acceptance criteria / edge cases.** Use `is None` checks rather than `or`. Test all three fields independently, including `False` as an explicit value for the timeout argument to illustrate that booleans are also integers in Python (document whether your API allows them).

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 06

In [11]:
def p06_configure(
    timeout: int | None = None,
    label: str | None = None,
    items: list[int] | None = None,
) -> tuple[int, str, list[int]]:
    resolved_timeout = 30 if timeout is None else timeout
    resolved_label = "untitled" if label is None else label
    resolved_items = [1, 2, 3] if items is None else items
    return resolved_timeout, resolved_label, resolved_items


def p06_broken(timeout: int | None) -> int:
    return timeout or 30

**Verification 06.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [12]:
assert p06_configure() == (30, "untitled", [1, 2, 3])
explicit_items: list[int] = []
assert p06_configure(0, "", explicit_items) == (0, "", [])
assert p06_configure(0, "", explicit_items)[2] is explicit_items
assert p06_configure(False)[0] is False  # Permissive API; strict validation is separate.
assert p06_broken(0) == 30  # Demonstrates the bug, not the desired behavior.
assert p06_configure(timeout=-1)[0] == -1
print("Problem 06: all checks passed")

Problem 06: all checks passed


## Problem 07 — Truthiness as a filter is not `None` filtering

**Challenge.** Implement two filters for `[None, 0, False, "", [], 4, "ok"]`: one removes only missing `None`, and the other removes every falsy element. Explain the difference by checking the exact outputs and identity of remaining values.

**Acceptance criteria / edge cases.** Do not compare with `!= None`; user-defined `__eq__` can have surprising semantics. Show why a truthy filter cannot be used as a missing-data filter.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 07

In [13]:
def p07_present(values: list[object]) -> list[object]:
    return [value for value in values if value is not None]


def p07_truthy(values: list[object]) -> list[object]:
    return [value for value in values if value]

**Verification 07.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [14]:
original: list[object] = [None, 0, False, "", [], 4, "ok"]
assert p07_present(original) == [0, False, "", [], 4, "ok"]
assert p07_truthy(original) == [4, "ok"]
assert p07_present(original)[2] is original[3]
assert len(p07_present(original)) == 6
assert len(p07_truthy(original)) == 2
print("Problem 07: all checks passed")

Problem 07: all checks passed


## Problem 08 — An explicit three-state decision

**Challenge.** Model `ALLOW`, `DENY`, and `UNKNOWN` using an enum. Known values should allow ordinary boolean checks; unknown must fail closed by raising `TypeError` rather than silently becoming true or false. Supply a `.resolve(default=...)` method for applications that deliberately choose an unknown fallback.

**Acceptance criteria / edge cases.** Check `bool(UNKNOWN)` raises, confirm `resolve` returns an actual bool, and ensure `ALLOW` and `DENY` behave predictably. Discuss why forcing the caller to choose a fallback is safer than silent conversion.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 08

In [15]:
from enum import Enum, auto


class P08Decision(Enum):
    ALLOW = auto()
    DENY = auto()
    UNKNOWN = auto()

    def __bool__(self) -> bool:
        if self is P08Decision.UNKNOWN:
            raise TypeError("UNKNOWN decision requires explicit resolution")
        return self is P08Decision.ALLOW

    def resolve(self, *, default: bool) -> bool:
        if type(default) is not bool:
            raise TypeError("default must be a bool")
        return default if self is P08Decision.UNKNOWN else bool(self)

**Verification 08.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [16]:
assert bool(P08Decision.ALLOW) is True
assert bool(P08Decision.DENY) is False
p03_expect_exception(TypeError, lambda: bool(P08Decision.UNKNOWN))
assert P08Decision.UNKNOWN.resolve(default=False) is False
assert P08Decision.UNKNOWN.resolve(default=True) is True
assert P08Decision.ALLOW.resolve(default=False) is True
assert P08Decision.DENY.resolve(default=True) is False
p03_expect_exception(TypeError, lambda: P08Decision.UNKNOWN.resolve(default=1))
print("Problem 08: all checks passed")

Problem 08: all checks passed


---

# Part III — Collections, iterators and aggregators

Problems 9–12 examine empty-input identities, one-shot streams, special-method lookup, and length hints.

## Problem 09 — `all`/`any` and short-circuit audits

**Challenge.** Build an iterable of truth probes. Determine the number and order of evaluations by `all()` and `any()`, including their behavior on empty input. Record results for sequences where an early false or true appears.

**Acceptance criteria / edge cases.** `all([])` is true; `any([])` is false. The aggregator short-circuits and does not call `__bool__` on later elements. Construct a fresh probe list for each invocation.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 09

In [17]:
def p09_probes(answers: list[bool]):
    events: list[int] = []

    class Probe:
        def __init__(self, position: int, answer: bool) -> None:
            self.position = position
            self.answer = answer

        def __bool__(self) -> bool:
            events.append(self.position)
            return self.answer

    return [Probe(i, value) for i, value in enumerate(answers)], events

**Verification 09.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [18]:
probes, events = p09_probes([True, False, True])
assert all(probes) is False
assert events == [0, 1]
probes, events = p09_probes([False, False, True, False])
assert any(probes) is True
assert events == [0, 1, 2]
probes, events = p09_probes([True, True])
assert all(probes) is True and events == [0, 1]
assert all([]) is True
assert any([]) is False
assert not any([])
print("Problem 09: all checks passed")

Problem 09: all checks passed


## Problem 10 — A generator is not an emptiness test

**Challenge.** Demonstrate that `bool(generator)` is true even when it has no elements. Implement a `Peekable` iterator for which truth means that at least one element remains. Repeated boolean checks must not lose data; zero-valued elements must still count as present.

**Acceptance criteria / edge cases.** Implement `__iter__`, `__next__`, and `__bool__` with a unique sentinel. A truth check may prefetch one item and cache it; repeated checks must not pull extra elements. Verify after exhaustion and with `None` and `0` values.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 10

In [19]:
from collections.abc import Iterable, Iterator


class P10Peekable(Iterator[object]):
    _EMPTY = object()

    def __init__(self, values: Iterable[object]) -> None:
        self._iterator = iter(values)
        self._cache: object = self._EMPTY
        self.pulls = 0

    def __iter__(self) -> "P10Peekable":
        return self

    def _fill(self) -> bool:
        if self._cache is not self._EMPTY:
            return True
        try:
            self._cache = next(self._iterator)
            self.pulls += 1
        except StopIteration:
            return False
        return True

    def __bool__(self) -> bool:
        return self._fill()

    def __next__(self) -> object:
        if self._cache is not self._EMPTY:
            result = self._cache
            self._cache = self._EMPTY
            return result
        result = next(self._iterator)
        self.pulls += 1
        return result

**Verification 10.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [20]:
assert bool(iter(())) is True  # Iterator objects have no implicit emptiness probe.
peek = P10Peekable([0, None, "done"])
assert bool(peek) is True and peek.pulls == 1
assert bool(peek) is True and peek.pulls == 1
assert next(peek) == 0 and peek.pulls == 1
assert bool(peek) is True and peek.pulls == 2
assert next(peek) is None
assert list(peek) == ["done"]
assert bool(peek) is False
p03_expect_exception(StopIteration, lambda: next(peek))
empty = P10Peekable([])
assert bool(empty) is False and empty.pulls == 0
print("Problem 10: all checks passed")

Problem 10: all checks passed


## Problem 11 — Why instance monkey-patching does not override `bool`

**Challenge.** Show that assigning an instance attribute named `__bool__` does not change the special-method dispatch used by `bool(instance)`. Then create a subclass that overrides `__bool__` at the **type** level and verify it is used.

**Acceptance criteria / edge cases.** Differentiate explicit `obj.__bool__()` attribute lookup from implicit `bool(obj)` protocol lookup; prove which function runs. This is an advanced CPython/Python data-model behavior, not a technique for ordinary API design.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 11

In [21]:
class P11Base:
    def __bool__(self) -> bool:
        return True


class P11Override(P11Base):
    def __bool__(self) -> bool:
        return False

**Verification 11.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [22]:
obj = P11Base()
obj.__bool__ = lambda: False  # Ordinary instance attribute shadows explicit lookup.
assert obj.__bool__() is False
assert bool(obj) is True  # Implicit special-method lookup uses the type.
assert bool(P11Override()) is False
assert P11Override().__bool__() is False
print("Problem 11: all checks passed")

Problem 11: all checks passed


## Problem 12 — `__length_hint__` is not `__len__`

**Challenge.** Write an iterator that exposes only `__length_hint__` but no `__len__`/`__bool__`. Its hint reports remaining items. Prove that `bool(iterator)` remains true even after exhaustion, whereas `list(iterator)` consumes the data. Add an explicit `has_remaining()` for this specific in-memory iterator.

**Acceptance criteria / edge cases.** A length hint is advisory, not a truth-value protocol. `len(iterator)` must raise `TypeError`; the iterator is intentionally not a collection.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 12

In [23]:
class P12HintedIterator:
    def __init__(self, values: list[int]) -> None:
        self._values = values.copy()
        self._index = 0

    def __iter__(self):
        return self

    def __next__(self) -> int:
        if self._index >= len(self._values):
            raise StopIteration
        result = self._values[self._index]
        self._index += 1
        return result

    def __length_hint__(self) -> int:
        return len(self._values) - self._index

    def has_remaining(self) -> bool:
        return self._index < len(self._values)

**Verification 12.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [24]:
hinted = P12HintedIterator([0, 2])
assert bool(hinted) is True
assert hinted.__length_hint__() == 2
p03_expect_exception(TypeError, lambda: len(hinted))
assert next(hinted) == 0
assert hinted.has_remaining() is True
assert list(hinted) == [2]
assert hinted.__length_hint__() == 0
assert hinted.has_remaining() is False
assert bool(hinted) is True
print("Problem 12: all checks passed")

Problem 12: all checks passed


---

# Part IV — Modeling truth deliberately

Problems 13–16 use dataclasses, currencies, expression trees, and task-state APIs to make truth meaning explicit.

## Problem 13 — Immutable numeric point with well-defined domain

**Challenge.** Make a 2D floating-point point where origin is false. Reject nonfinite coordinates on construction (especially NaN, whose nonzero comparisons can surprise readers). Offer an explicit squared-distance method, and prove small nonzero values are true without arbitrary rounding.

**Acceptance criteria / edge cases.** Only built-in int/float coordinates are accepted (excluding bool). Reject infinities and NaN with `ValueError`. Do not compute a square root merely to check origin.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 13

In [25]:
from dataclasses import dataclass
from math import isfinite


@dataclass(frozen=True)
class P13FinitePoint:
    x: float
    y: float

    def __post_init__(self) -> None:
        for coordinate in (self.x, self.y):
            if type(coordinate) not in (int, float):
                raise TypeError("coordinates must be int or float, not bool")
            if not isfinite(coordinate):
                raise ValueError("coordinates must be finite")

    def __bool__(self) -> bool:
        return self.x != 0 or self.y != 0

    def distance_squared(self) -> float:
        return self.x * self.x + self.y * self.y

**Verification 13.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [26]:
assert bool(P13FinitePoint(0.0, -0.0)) is False
assert bool(P13FinitePoint(1e-200, 0.0)) is True
assert bool(P13FinitePoint(-2, 3)) is True
assert P13FinitePoint(3, 4).distance_squared() == 25
p03_expect_exception(ValueError, lambda: P13FinitePoint(float("nan"), 0))
p03_expect_exception(ValueError, lambda: P13FinitePoint(0, float("inf")))
p03_expect_exception(TypeError, lambda: P13FinitePoint(True, 0))
print("Problem 13: all checks passed")

Problem 13: all checks passed


## Problem 14 — Money: zero vs negative amounts

**Challenge.** Create immutable `Money` with an exact `Decimal` amount and a currency code. Boolean conversion should mean **nonzero**, including a negative debt; comparison with zero should not be confused with whether funds are positive. Addition must reject mixed currencies.

**Acceptance criteria / edge cases.** Do not silently convert floats to decimal. Accept only finite Decimal amounts, positive/negative and signed zero; currency must be a three-letter uppercase ASCII code. Test `has_positive_balance()` separately.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 14

In [27]:
from dataclasses import dataclass
from decimal import Decimal


@dataclass(frozen=True)
class P14Money:
    amount: Decimal
    currency: str

    def __post_init__(self) -> None:
        if not isinstance(self.amount, Decimal):
            raise TypeError("pass a Decimal amount explicitly")
        if not self.amount.is_finite():
            raise ValueError("amount must be finite")
        if len(self.currency) != 3 or not all("A" <= c <= "Z" for c in self.currency):
            raise ValueError("currency must be three uppercase ASCII letters")

    def __bool__(self) -> bool:
        return self.amount != Decimal(0)

    def has_positive_balance(self) -> bool:
        return self.amount > Decimal(0)

    def __add__(self, other: "P14Money") -> "P14Money":
        if not isinstance(other, P14Money):
            return NotImplemented
        if self.currency != other.currency:
            raise ValueError("cannot add different currencies")
        return P14Money(self.amount + other.amount, self.currency)

**Verification 14.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [28]:
zero = P14Money(Decimal("-0.00"), "USD")
debt = P14Money(Decimal("-2.50"), "USD")
credit = P14Money(Decimal("2.50"), "USD")
assert bool(zero) is False and bool(debt) is True
assert debt.has_positive_balance() is False
assert credit.has_positive_balance() is True
assert bool(debt + credit) is False
assert (debt + credit).currency == "USD"
p03_expect_exception(ValueError, lambda: debt + P14Money(Decimal("1"), "EUR"))
p03_expect_exception(TypeError, lambda: P14Money(1.25, "USD"))
p03_expect_exception(ValueError, lambda: P14Money(Decimal("NaN"), "USD"))
print("Problem 14: all checks passed")

Problem 14: all checks passed


## Problem 15 — Expression objects must reject implicit truth

**Challenge.** Implement a composable predicate expression for records. `&`, `|`, and `~` create expression trees; `and`, `or`, and `if expression` must **not** silently treat the query as true. Use `__bool__` to raise `TypeError` and an explicit `.matches(record)` API to evaluate.

**Acceptance criteria / edge cases.** Make `&` and `|` check operand type, represent nodes in a readable way, and short-circuit actual record evaluation. Confirm that Python's `and` attempts a forbidden truth test rather than invoking `__and__`.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 15

In [29]:
from collections.abc import Callable, Mapping


class P15Predicate:
    def __init__(self, evaluator: Callable[[Mapping[str, object]], bool], description: str):
        self._evaluator = evaluator
        self.description = description

    def __bool__(self) -> bool:
        raise TypeError("A predicate is an expression; use .matches(record)")

    def matches(self, record: Mapping[str, object]) -> bool:
        answer = self._evaluator(record)
        if type(answer) is not bool:
            raise TypeError("predicate evaluator must return bool")
        return answer

    def __and__(self, other: "P15Predicate") -> "P15Predicate":
        if not isinstance(other, P15Predicate):
            return NotImplemented
        return P15Predicate(
            lambda record: self.matches(record) and other.matches(record),
            f"({self.description} & {other.description})",
        )

    def __or__(self, other: "P15Predicate") -> "P15Predicate":
        if not isinstance(other, P15Predicate):
            return NotImplemented
        return P15Predicate(
            lambda record: self.matches(record) or other.matches(record),
            f"({self.description} | {other.description})",
        )

    def __invert__(self) -> "P15Predicate":
        return P15Predicate(lambda record: not self.matches(record), f"~{self.description}")

**Verification 15.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [30]:
events: list[str] = []
active = P15Predicate(lambda row: events.append("active") is None and row["active"] is True, "active")
large = P15Predicate(lambda row: events.append("large") is None and row["amount"] > 100, "large")
combined = active & large
assert combined.matches({"active": True, "amount": 200}) is True
assert events == ["active", "large"]
events.clear()
assert combined.matches({"active": False, "amount": 200}) is False
assert events == ["active"]  # `and` in the evaluator short-circuits.
assert (active | large).matches({"active": False, "amount": 200}) is True
assert (~active).matches({"active": False, "amount": 200}) is True
p03_expect_exception(TypeError, lambda: bool(combined))
p03_expect_exception(TypeError, lambda: active and large)
p03_expect_exception(TypeError, lambda: combined & True)
assert "&" in combined.description
print("Problem 15: all checks passed")

Problem 15: all checks passed


## Problem 16 — Asynchronous-style operation with ambiguous truth

**Challenge.** Implement a tiny stateful operation with `PENDING`, `SUCCEEDED`, `FAILED`. A truth test on the operation is ambiguous (done? successful? has result?), so it should always raise. Provide explicit `.done()`, `.succeeded()`, and `.result()` methods with meaningful errors.

**Acceptance criteria / edge cases.** Validate legal transitions: completion once only, failure requires an exception, and fetching a result before completion or after failure raises appropriately. A successful result of `0` remains a successful operation.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 16

In [31]:
from enum import Enum, auto


class P16State(Enum):
    PENDING = auto()
    SUCCEEDED = auto()
    FAILED = auto()


class P16Operation:
    def __init__(self) -> None:
        self._state = P16State.PENDING
        self._value: object = None
        self._error: Exception | None = None

    def __bool__(self) -> bool:
        raise TypeError("Operation truth is ambiguous; call .done() or .succeeded()")

    def done(self) -> bool:
        return self._state is not P16State.PENDING

    def succeeded(self) -> bool:
        return self._state is P16State.SUCCEEDED

    def complete(self, value: object) -> None:
        if self.done():
            raise RuntimeError("already completed")
        self._value = value
        self._state = P16State.SUCCEEDED

    def fail(self, error: Exception) -> None:
        if self.done():
            raise RuntimeError("already completed")
        if not isinstance(error, Exception):
            raise TypeError("error must be an Exception instance")
        self._error = error
        self._state = P16State.FAILED

    def result(self) -> object:
        if not self.done():
            raise RuntimeError("operation is pending")
        if self._state is P16State.FAILED:
            assert self._error is not None
            raise self._error
        return self._value

**Verification 16.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [32]:
pending = P16Operation()
assert pending.done() is False and pending.succeeded() is False
p03_expect_exception(TypeError, lambda: bool(pending))
p03_expect_exception(RuntimeError, pending.result)
pending.complete(0)
assert pending.done() is True and pending.succeeded() is True
assert pending.result() == 0
p03_expect_exception(RuntimeError, lambda: pending.complete(1))
failed = P16Operation()
p03_expect_exception(TypeError, lambda: failed.fail("bad"))
failed.fail(ValueError("failure from task"))
assert failed.done() is True and failed.succeeded() is False
p03_expect_exception(ValueError, failed.result)
p03_expect_exception(TypeError, lambda: bool(failed))
print("Problem 16: all checks passed")

Problem 16: all checks passed


---

# Part V — Inheritance, graphs and robust testing

Problems 17–20 combine method-resolution subtleties, cycle safety, metamorphic testing, and an integrated queue capstone.

## Problem 17 — Inherited `__bool__` beats a new `__len__`

**Challenge.** A base class defines `__bool__` using an enabled flag. A derived collection defines `__len__` but not `__bool__`. Predict truth for an empty enabled object and a nonempty disabled object. Then fix the derived class so truth consistently means nonempty.

**Acceptance criteria / edge cases.** Show that method-resolution order includes inherited `__bool__`. Do not assume that defining `__len__` shadows the base hook. The override should return an explicit bool.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 17

In [33]:
class P17EnabledBase:
    def __init__(self, enabled: bool) -> None:
        self.enabled = enabled

    def __bool__(self) -> bool:
        return self.enabled


class P17Surprising(P17EnabledBase):
    def __init__(self, enabled: bool, values: list[int]) -> None:
        super().__init__(enabled)
        self.values = values

    def __len__(self) -> int:
        return len(self.values)


class P17Fixed(P17Surprising):
    def __bool__(self) -> bool:
        return len(self) != 0

**Verification 17.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [34]:
assert len(P17Surprising(True, [])) == 0
assert bool(P17Surprising(True, [])) is True
assert bool(P17Surprising(False, [1])) is False
assert bool(P17Fixed(True, [])) is False
assert bool(P17Fixed(False, [1])) is True
assert type(P17Fixed(False, [1]).__bool__()) is bool
print("Problem 17: all checks passed")

Problem 17: all checks passed


## Problem 18 — Cycle-safe graph reachability via `__bool__`

**Challenge.** Model a mutable graph node as true if *any reachable node* is marked active, including itself. Your implementation must tolerate cycles and avoid recursive stack overflow from deep graphs; use identity-based visited tracking.

**Acceptance criteria / edge cases.** Two separate nodes can share the same display name; do not identify nodes by name. Demonstrate both a directed cycle and a large chain. This is an intentionally custom domain meaning, **not** normal collection emptiness.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 18

In [35]:
class P18Node:
    def __init__(self, name: str, active: bool = False) -> None:
        self.name = name
        self.active = active
        self.neighbors: list["P18Node"] = []

    def connect(self, other: "P18Node") -> None:
        self.neighbors.append(other)

    def __bool__(self) -> bool:
        pending = [self]
        visited: set[int] = set()
        while pending:
            node = pending.pop()
            identity = id(node)
            if identity in visited:
                continue
            visited.add(identity)
            if node.active:
                return True
            pending.extend(node.neighbors)
        return False

**Verification 18.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [36]:
a = P18Node("same")
b = P18Node("same")
c = P18Node("target")
a.connect(b)
b.connect(a)
b.connect(c)
assert bool(a) is False
c.active = True
assert bool(a) is True and bool(b) is True
assert bool(P18Node("other")) is False
chain = [P18Node(str(i)) for i in range(1500)]
for previous, following in zip(chain, chain[1:]):
    previous.connect(following)
chain[-1].active = True
assert bool(chain[0]) is True  # Iterative walk avoids recursion limits.
print("Problem 18: all checks passed")

Problem 18: all checks passed


## Problem 19 — Property-style randomized verification

**Challenge.** For a mutable queue, the invariant `bool(queue) == (len(queue) > 0)` should hold after **every** mutation. Use a seeded generator, a reference `collections.deque`, and a long sequence of operations. Include falsy payloads such as `None`, `0`, and `False`.

**Acceptance criteria / edge cases.** No third-party property-testing packages are needed. Check both content and length against the reference after every step; a fixed seed makes any failing sequence reproducible. Test empty-pop behavior explicitly.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 19

In [37]:
from collections import deque
from random import Random


class P19Queue:
    def __init__(self) -> None:
        self._items: deque[object] = deque()

    def enqueue(self, item: object) -> None:
        self._items.append(item)

    def dequeue(self) -> object:
        if not self._items:
            raise IndexError("dequeue from empty queue")
        return self._items.popleft()

    def __len__(self) -> int:
        return len(self._items)

    def snapshot(self) -> list[object]:
        return list(self._items)

**Verification 19.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [38]:
rng = Random(20260920)
queue = P19Queue()
reference: deque[object] = deque()
payloads: list[object] = [None, 0, False, "", [], 1, "payload"]
for step in range(600):
    if not reference or rng.randrange(3) != 0:
        value = payloads[rng.randrange(len(payloads))]
        queue.enqueue(value)
        reference.append(value)
    else:
        assert queue.dequeue() == reference.popleft()
    assert len(queue) == len(reference), f"length diverged at step {step}"
    assert bool(queue) is (len(reference) > 0), f"truth diverged at step {step}"
    assert queue.snapshot() == list(reference)
while reference:
    assert queue.dequeue() == reference.popleft()
assert bool(queue) is False
p03_expect_exception(IndexError, queue.dequeue)
print("Problem 19: all checks passed")

Problem 19: all checks passed


## Problem 20 — Capstone: a robust task scheduler

**Challenge.** Build a scheduler with queued tasks having `priority` (integer), `payload` (any object, including falsy values), and stable FIFO ordering among equal priorities. Scheduler truth must mean **tasks are queued**. Implement `submit`, `run_next`, `__len__`, and `drain`; don't accidentally discard payload `0`/`False`/`None`, and don't confuse a completed result with queue emptiness.

**Acceptance criteria / edge cases.** Use a heap of `(priority, insertion_counter, callable)` tuples; lower number means earlier execution. Check non-bool integer priority, ensure tasks are callable, propagate task errors without replaying a popped task, and return `None` as a legitimate result from `run_next`. Empty `run_next` raises `IndexError`.

Try implementing it before revealing the solution cell below. Each solution is self-contained within the notebook; run cells from top to bottom.

### Solution 20

In [39]:
from collections.abc import Callable, Iterator
import heapq
from itertools import count


class P20Scheduler:
    def __init__(self) -> None:
        self._tasks: list[tuple[int, int, Callable[[], object]]] = []
        self._sequence = count()

    def submit(self, priority: int, task: Callable[[], object]) -> None:
        if type(priority) is not int:
            raise TypeError("priority must be int (not bool)")
        if not callable(task):
            raise TypeError("task must be callable")
        heapq.heappush(self._tasks, (priority, next(self._sequence), task))

    def __len__(self) -> int:
        return len(self._tasks)

    def run_next(self) -> object:
        if not self._tasks:  # Direct check avoids reinterpreting a task's result.
            raise IndexError("scheduler has no queued tasks")
        _, _, task = heapq.heappop(self._tasks)
        return task()

    def drain(self) -> Iterator[object]:
        while self:  # Delegates to __len__; does not inspect returned payloads.
            yield self.run_next()

**Verification 20.** Execute the assertions; a passing cell prints a confirmation. Assertions intentionally cover non-happy paths.

In [40]:
scheduler = P20Scheduler()
assert bool(scheduler) is False
p03_expect_exception(IndexError, scheduler.run_next)
p03_expect_exception(TypeError, lambda: scheduler.submit(True, lambda: 1))
p03_expect_exception(TypeError, lambda: scheduler.submit(0, 42))
executed: list[str] = []
scheduler.submit(2, lambda: (executed.append("later"), False)[1])
scheduler.submit(1, lambda: (executed.append("first"), 0)[1])
scheduler.submit(1, lambda: (executed.append("second"), None)[1])
assert bool(scheduler) is True and len(scheduler) == 3
assert list(scheduler.drain()) == [0, None, False]
assert executed == ["first", "second", "later"]
assert bool(scheduler) is False and len(scheduler) == 0

failing = P20Scheduler()
def p20_boom():
    raise ValueError("task failed")
failing.submit(0, p20_boom)
p03_expect_exception(ValueError, failing.run_next)
assert bool(failing) is False  # Failed task popped exactly once.
print("Problem 20: all checks passed")

Problem 20: all checks passed


---

# Final synthesis — rules to retain

1. Define only `__len__` for collection-style emptiness when appropriate; define `__bool__` for an intentional, documented alternative meaning. When both exist (including via inheritance), `__bool__` wins.
2. `__bool__` must return `True` or `False` as a real `bool`; `self.x or self.y` can return an integer. Bad `__len__` return types, negative values, or lengths larger than platform sequence-size limits raise errors.
3. `and`/`or` return operand objects and short-circuit. `not` always returns `bool`. `all([])` is `True`, while `any([])` is `False`.
4. An empty generator or exhausted iterator is not automatically falsy. `__length_hint__` is **not** an emptiness hook. A peekable iterator may make a boolean test advance its source; document this side effect.
5. Use `is None` for missing-data checks; falsy values like `0`, `False`, and `[]` may be fully valid. For expressions, operations, or unknown states with ambiguous truth, make callers use explicit APIs.
6. Check invariants after mutations, negative tests, stable ordering, and short-circuit behavior. Prefer deterministic, dependency-free tests that run with **Restart & Run All**.

**Optional further challenges:** implement mutation-safe iteration for Problem 4; design a peekable **async** iterator without synchronous truthiness for Problem 10; add cancellation and retries to Problem 20 while documenting what `bool(scheduler)` means. These are exercises for the reader, not untested claimed solutions.

**Scope:** The basic dispatch and Point examples follow the user-provided Boolean lesson; the more elaborate exercises and best-practice recommendations are original extensions.